# Universal Turing Machines 

In this example, we'll be explaining the design of a Universal Turing Machine (UTM), which can take in any input Turing Machine $T$ and string $w$, and fully simulate the behavior of $M$ on $w$.

## Example Turing Machine and String
Consider the following Turing Machine, $T$, which adds 1 to a binary number that is represented in reverse (least significant bit on the left).

In [11]:
from automata.tm.dtm import DTM

tm = DTM(
    states={'q0', 'q_accept'},
    input_symbols={'0', '1'},
    tape_symbols={'0', '1', '_'},
    transitions={
        # Start in q0: add 1 from least significant bit (left)
        'q0': {
            '0': ('q_accept', '1', 'N'),  # 0 + 1 = 1, no carry
            '1': ('q0', '0', 'R'),        # 1 + 1 = 0 (carry), move right
            '_': ('q_accept', '1', 'N'),  # If carry still remains, write 1
        },
    },
    initial_state='q0',
    blank_symbol='_',
    final_states={'q_accept'}
)

Using `graphviz`, we can visualize the following states in $T$:

In [10]:
from graphviz import Digraph

def visualize_tm_graphviz(transitions, filename='tm_graph'):
    dot = Digraph(format='pdf')
    
    # Global graph settings for spacing and aesthetics
    dot.attr(ranksep='1.5', nodesep='1.0')  # more space between rows/columns
    dot.attr('node', shape='circle', fontsize='14', width='0.5', fixedsize='false')
    dot.attr('edge', fontsize='12', penwidth='1.2')

    for src, symbol_map in transitions.items():
        dot.node(src)  # make sure all states are defined
        for read, (dst, write, move) in symbol_map.items():
            label = f"  {read}→{write},{move}"
            dot.edge(src, dst, label=label)

    dot.render(filename, view=True)

# Example usage:
transitions = {
    'q0': {
        '0': ('q_accept', '1', 'N'),
        '1': ('q0', '0', 'R'),
        '_': ('q_accept', '1', 'N'),
    }
}

visualize_tm_graphviz(transitions)


Now, consider the string $w=1101$, which represents a reversed $11$ in binary $(1011)$. We expect an output of a reversed $12$ in binary, or $0011$.

In [9]:
# Example run
progress = tm.read_input_stepwise("1101") # Reverse binary for 1011 (11)
for i in progress:
    print(i)

TMConfiguration('q0', TMTape('1101', '_', 0))
TMConfiguration('q0', TMTape('0101', '_', 1))
TMConfiguration('q0', TMTape('0001', '_', 2))
TMConfiguration('q_accept', TMTape('0011', '_', 2))


## Encoding a Turing Machine and String
### Encoding Schema
To encode a Turing Machine $T=(Q, \sum, \Gamma, \delta, q_0, h_a, h_r)$ as a binary string, we first must encode the individual parts of each transition. This includes the symbols in the alphabets, the states, and the tape-head directions. These encodings will then be used to encode the transitions and the input string.

Our schema involves an ordering function, ORD that assigns a unique-to-the-states positive integer to each state, a unique-to-the-alphabets positive integer to each alphabet symbol, and assigns a unique-to-the-directions positive integer to each direction. Then, each state, symbol, and direction is encoded in unary by the function ENCODE(x)=$1^{ORD(x)}$, a length of $1$ s equal to its ORD value.

In [ ]:
def generate_encoding_schema(tm):
    states = tm.states.union({"q_reject"})
    tape_alphabet = tm.tape_symbols

    directions = ['R', 'L', 'N']

    state_enc = {state: "1"*(i+1) for i, state in enumerate(states)}
    tape_enc = {letter: "1"*(i+1) for i, letter in enumerate(tape_alphabet)}
    dir_enc = {direction: "1"*(i+1) for i, direction in enumerate(directions)}

    return state_enc, tape_enc, dir_enc

schema = generate_encoding_schema(tm)

Using this schema, the following table encodes $T$:

<img src="UTM.png" width="50%" height="auto">


### String Encoding
A string $w=w_1...w_n$ is encoded by separating the character encodings with a $0$:<br>

$\begin{align}
ENCODE(w)&=01^{ORD(w_1)}01^{ORD(w_2)}...01^{ORD(w_n)}\end{align}$

In [20]:
def encode_string(string, schema):
    _, tape_enc, _ = schema

    encoded_string = ""

    for char in string:
        encoded_string += f"0{tape_enc[char]}"

    return encoded_string

encode_string("1101", schema)

'0110110111011'

For our example string of $w=1101$, the encoded string (using the same ordering scheme
as above) looks like:<br>

$\begin{align}
ENCODE(1101)&=0110110111011\end{align}$

### Transition Encoding
Suppose that we have the transition δ(p, a) = (q, b, D) that looks like this:

The transition is encoded by separating the encodings of the individual parts of the transition with 0s. Our convention is to order the parts of the transition in the same order they appear in the functional representation $\delta(p, a) = (q, b, D)$. Thus, the generic transition shown above becomes:

$\begin{align}
ENCODE(1101)&=1^{ORD(p)}01^{ORD(a)}01^{ORD(q)}01^{ORD(b)}01^{ORD(D)}\end{align}$

In [ ]:
def encode_TM(tm, schema):

    transitions = ""
    state_enc, tape_enc, dir_enc = schema

    for src, symbol_map in tm.transitions.items():
        for read, (dst, write, move) in symbol_map.items():
            transitions += f"{state_enc[src]}0{tape_enc[read]}0{state_enc[dst]}0{tape_enc[write]}0{dir_enc[move]}00"
    return transitions


encode_TM(tm, schema)

'1110111010110111001110110111011101001110101011011100'

Consider the first transistion $t_1$ in our instance, $\delta(q_0, 0)=(q_{accept}, 1, S)$. This encoding is given by:
$\begin{align}
ENCODE(t_1)&=111011101011110111\end{align}$

### TM Encoding
A Turing Machine can be represented by its transition function alone. Thus, to encode an entire Turing Machine $T$ with transitions $t_1$, $t_2$, up through $t_k$, we simply add a $0$ between adjacent pairs of encoded transitions. The transitions can appear in any order. The encoding of $T$ is then:

$\begin{align}
ENCODE(T)&=ENCODE(t_1)0ENCODE(t_2)0...ENCODE(t_n)0\end{align}$

Since the encoding of each transition itself ends in a $0$, transitions are identifiable as being the substrings between closest pairs of the sub-string $00$. There are no other occurences of $00$ in ENCODE($T$). Notice that the last transition ends with $00$, which means that ENCODE($T$) ends with $00$.


### Encoding a TM and its String
The input to a UTM requires both an input TM $T$ and an input string $w$. To encode these together, we simply concatenate their binary
encodings:
$\begin{align} ENCODE(T)ENCODE(w) \end{align}$

Notice that since the encoding of $T$ ends in two 0s and the encoding of the string $w$ begins with a $0$,
we can easily find the beginning of $w$ by scanning right until we find the sub-string $000$; there are no other
places in the entire encoding that have three consecutive 0s.

## The Universal Turing Machine
The UTM typically has an initialization and simulation phase. The initialization phase involves encoding the TM using the process described above and breaking it down into the following three tapes used to simulate the TM.


After the initialization phase, the UTM implementation will have three tapes:
* Tape 1 has the encoded input string $w$
* Tape 2 has the encoded TM $T$
* Tape 3 keeps track of the current state $q$

Note that at the end of the initialization phase, all tapes have their pointers reset back to index 0.

For our instance, the tapes have the following values:

In [ ]:
class UTM:
    def __init__(self, tm, string):
        self.tm = tm
        self.string = string

        self.schema = generate_encoding_schema(tm)

        self.tapes = self.create_UTM_tapes()
        self.pointers = [0,0,0] # the index of the pointer on each tape

    def create_UTM_tapes(self):
        tape_1 = encode_string(self.string, self.schema)[1:]
        tape_2 = encode_TM(self.tm, self.schema)  # <-- fixed
        state_enc, tape_enc, dir_enc = self.schema
        tape_3 = state_enc[self.tm.initial_state]
        return tape_1, tape_2, tape_3
    
    def execute_transition_abbreviated(self):
        t1, t2, t3 = self.tapes
        p1, p2, p3 = self.pointers

        # search tape 1 for a state match on tape 3
        transitions = t2.split("00")[:-1]
        start_states = [t.split("0")[0] for t in transitions]
        print(f"Start states: {start_states}")
        current_state = t3
        

    
    def execute_transition(self):
        # executes a single transition

        # Search tape 1 for a transition with first component equal to the string of 1s on tape 3
        t1, t2, t3 = self.tapes
        p1, p2, p3 = self.pointers

        while True:
            try:
                # check to see if the current transition equals the state
                for i in range(len(t3)):

                    # if it doesn't, reset the state pointer to 0 to check the next transition
                    if t2[p2] != t3[p3]:
                        p3 = 0
                        while t2[p2] != "0":
                            p2 += 1 
                        p2 += 1 
                        break
                    p2 += 1
                    p3 += 1
                
                # if we read the entire state, we may still need to check if the state matches 
                else:
                    if t2[p2] != "0": # we didn't read the entire state
                        continue
                    else: # we found a state match, time to check characters from the input string
                        p3 = 0 # cleanup
                        p2 += 1 # skip the delimiter
            except:
                pass

    def __str__(self):
        s = "Universal Turing Machine\n"
        for i, tape in enumerate(self.tapes):
            s += f"T{i+1}: {tape} at position {self.pointers[i]}\n"
        return s

utm = UTM(tm, "1101")
print(utm)
utm.execute_transition_abbreviated()

Universal Turing Machine
T1: 110110111011 at position 0
T2: 1110111010110111001110110111011101001110101011011100 at position 0
T3: 111 at position 0

Start states: ['111', '111', '111']


## Simulation
For the first transition, we show each step of the TM's execution explicitly, then at a higher level later on.

* Search tape 1 for a transition with first component equal to the string of 1s on tape 3, and second
component equal to the string of 1s at the current position on tape 2.

<img src="UTM_binary.png" width="50%" height="auto">
